# **HMAC**


This notebook implements HMAC from scratch in Python. HMAC can be implemented in a variety of ways depending on the chosen hashing algorithm, but the most common one is the SHA-256 algorithm, which is used in this implementation.

In [1]:
import hmac  # to verify correct implementation
from hashlib import sha256  # chosen hashing algorithm

The secret key is provided by the user as a regular string value (e.g. "my-secret-key"). The HMAC algorigthm requires the key to be the same size as the block-size of the chosen algorithm. 

As the SHA-256 algorithm was chosen, the key needs to be 64 bytes in size. The easiest way to achieve this in Python is to first encode it to utf-8 as a `bytes` object, and then pad it with empty individual bytes until it is 64 bytes in size. 

In [2]:
def prepare_key(key_string:str) -> bytes:
    """
    Make sure the key is encoded to utf-8
    Make sure the key is 64 bytes in size (required for ...)
    """
    key = key_string.encode('utf-8')
    if len(key) > 64:
        key = sha256(key).digest()
    else:
        delta = 64 - len(key)
        key += b'\x00'*delta
    return key

THe HMAC algorithm itself is relatively simple. It performs a set of nested hashes on a combination of the message (`text`) and the key (`k0`)

In [3]:
def xor(a:bytes, b:bytes) -> bytes:
    """Perform xor operation on a per-byte level of two bytes objects"""
    return bytes(x ^ y for x,y in zip(a, b))


def HMAC(key:str, message:str) -> str:
    """
    Computes the HMAC hexidecimal string from a given key and message
    """
    IPAD = b'\x36'*64
    OPAD = b'\x5c'*64
    text = message.encode('utf-8')

    k0 = prepare_key(key)

    output = sha256(
        xor(k0, OPAD)
        + sha256(
            xor(k0, IPAD) + text
        ).digest()
    ).hexdigest()

    return output


Here we verify that our implementation matches the python native implementation

In [4]:
message = "hello"
key = "my-secret-key"

h1 = HMAC(key, message)
h2 = hmac.new(key.encode('utf-8'), message.encode('utf-8'), sha256).hexdigest()

print(f'HMAC from scratch: \t {h1}')
print(f'HMAC Python native: \t {h2}')

HMAC from scratch: 	 3ac7fc22f5cbd1aaaea11fac013d3e6f5d894fe2fc51ab98b6d3b1593a45255a
HMAC Python native: 	 3ac7fc22f5cbd1aaaea11fac013d3e6f5d894fe2fc51ab98b6d3b1593a45255a
